# Task 1 (Chapter 8) — Leakage-Free Retrain: Colab Run

Trains a single config — **MAnet + MiT-B2, ImageNet weights** — on the 3,248 Task 1 images that remain after excluding all 228 PIID Stage-1 images (see `more_experiments/manifests/piid_overlap_report.json`). This is the only Task-1 checkpoint allowed to generate localisation info for the rest of the Chapter 8 experiments — it must not be mixed up with the original `task_1/results/.../manet__mit_b2__imagenet_fold*.pt` checkpoints, which were trained on PIID-contaminated data.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the dataset

Same raw `wound-region-segmentation` dataset as Task 1 — filtering down to the leakage-free subset happens via the manifest filelist in step 6, not via a different zip.

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-region-segmentation.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-region-segmentation"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
source = candidates[0]
print("dataset found at:", source)

os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(source, DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Ensure the `skin` color mapping matches training code

Forces `skin` to `#0000FF` in the dataset's `color-mappings.json` (matching `datasets/README.md`) — a no-op if it's already correct.

In [ ]:
import json

color_map_path = f"{DATASET_DIR}/color-mappings.json"
with open(color_map_path) as f:
    mapping = json.load(f)
print("before:", mapping)

if mapping.get("skin") != "#0000FF":
    mapping["skin"] = "#0000FF"
    with open(color_map_path, "w") as f:
        json.dump(mapping, f, indent=4)
    print("fixed ->", mapping)
else:
    print("already correct, no change")

## 5. Install dependencies

Same `requirements.txt` as Task 1 — `run_leakagefree.py` imports `task_1/src` directly rather than duplicating it.

In [ ]:
import os
!pip install -q -r {REPO_DIR}/task_1/requirements.txt
os.chdir(f"{REPO_DIR}/more_experiments/task1_retrain")
print(os.getcwd())

## 6. Regenerate + verify the leakage-free manifest

Re-derives the PIID overlap from *this* copy of the dataset (not just trusting the committed manifest) — writes `../manifests/task1_leakagefree_filelist.txt` (expected: 3,248 files) and `../manifests/task1_excluded_piid_stage1.txt` (expected: 228 files). `run_leakagefree.py` also asserts no PIID file is present in the filelist before training starts, as a second safety net.

In [ ]:
!python ../scripts/verify_piid_overlap.py

## 7. Redirect checkpoints/results to Drive

So a killed/disconnected runtime doesn't lose completed-fold history or checkpoints.

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/task1_leakagefree_outputs"

with open("configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/leakagefree_results.csv"
with open("configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 8. Train — MAnet + MiT-B2, 3-fold, leakage-free

Single config, no grid. `run_leakagefree.py` already skips completed `(config, fold)` rows and resumes a killed fold from its last saved epoch — but a crash still kills the Python process itself, so this loop restarts it automatically. Output is streamed line-by-line (`Popen` + `-u`). A full runtime disconnect kills the loop too (Colab tears down the kernel); at that point just re-run this cell — training picks up from the last saved epoch either way.

In [ ]:
import subprocess
import time

cmd = ["python", "-u", "run_leakagefree.py", "--batch", "64", "--workers", "16"]
while True:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    returncode = proc.wait()
    if returncode == 0:
        print("training finished")
        break
    print(f"run_leakagefree.py exited with code {returncode}, retrying in 10s...")
    time.sleep(10)

## 9. Preview sample panels

After every completed fold, `run_leakagefree.py` saves a 3-panel PNG (image / ground truth / prediction overlay) to `results/samples/`.

In [ ]:
import glob
from IPython.display import Image, display

samples_dir = f"{OUTPUT_DIR}/results/samples"
paths = sorted(glob.glob(f"{samples_dir}/*.png"))
print(f"{len(paths)} sample panels")
for path in paths:
    print(path.split("/")[-1])
    display(Image(filename=path))

## 10. Check results

3 rows expected (one per fold) once training completes.

In [ ]:
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/results/leakagefree_results.csv")
display(df)
print("mean test mIoU:", df["miou"].mean())